# Multi-turn policy comparison (calming vs. provocative)

Runs short 5-turn conversations across all seven emotions with three deterministic policies (calming, provocative, always-validate baseline). Saves per-turn logs, summaries, and heatmaps under `results/multiturn_runs/`. Set `OPENAI_API_KEY` before running; GPU is optional but recommended.

In [1]:
!pip uninstall -y dynamic-conversation dynamic_conversation
!pip cache purge
!pip install --no-cache-dir git+https://github.com/Javin-Mendiratta/Dynamic-Conversation.git@derek_12_13


from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get('OpenAI')

from pathlib import Path
import json
import pandas as pd
from dynamic_conversation import (
    EmotionFlowAnalyzer,
    MultiTurnRollout,
    calming_policy,
    provocative_policy,
    always_validate_policy,
)

from pathlib import Path

Found existing installation: dynamic-conversation 0.1.1
Uninstalling dynamic-conversation-0.1.1:
  Successfully uninstalled dynamic-conversation-0.1.1
Files removed: 12
  Cloning https://github.com/Javin-Mendiratta/Dynamic-Conversation.git (to revision derek_12_13) to /tmp/pip-req-build-x7lbuts6
  Running command git clone --filter=blob:none --quiet https://github.com/Javin-Mendiratta/Dynamic-Conversation.git /tmp/pip-req-build-x7lbuts6
  Running command git checkout -b derek_12_13 --track origin/derek_12_13
  Switched to a new branch 'derek_12_13'
  Branch 'derek_12_13' set up to track remote branch 'derek_12_13' from 'origin'.
  Resolved https://github.com/Javin-Mendiratta/Dynamic-Conversation.git to commit 1d72a750104706db99b51c6f4fd8089d9da0d186
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for dynamic-conversation: filename=dynamic_conversation-0.1.1-py3-none-any.whl size=26476

In [2]:
# Configuration
emotions = EmotionFlowAnalyzer.EMOTIONS
policies = {
    "calming": calming_policy,
    "provocative": provocative_policy,
    "validate": always_validate_policy,
}
runs_per_emotion = 3  # keep small to finish within ~20–30 minutes
turns = 5
style_modifier = "concise and emotionally attuned"
out_dir = Path("results/multiturn_runs")
out_dir.mkdir(parents=True, exist_ok=True)

sim = MultiTurnRollout(
    use_gpu=True,
    turns=turns,
    default_style=style_modifier,
)
print(f"Device set to GPU? {sim.emotion_analyzer.classifier.device != -1}")

Loading emotion classifier: j-hartmann/emotion-english-distilroberta-base


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cuda:0


Device set to GPU? True


In [3]:
# Run all policies
per_policy = {}
for name, fn in policies.items():
    csv_path = out_dir / f"{name}_5turn.csv"
    plot_path = out_dir / f"{name}_5turn_heatmap.png"
    per_turn_df, summary_df = sim.run_policy_batch(
        policy_name=name,
        policy_fn=fn,
        emotions=emotions,
        runs_per_emotion=runs_per_emotion,
        turns=turns,
        style_modifier=style_modifier,
        use_llm_seed=True,
        use_esconv_seed=False,
        save_csv=csv_path,
        save_plot=plot_path,
    )
    per_policy[name] = (per_turn_df, summary_df)
    display(summary_df.head())
print("✓ Completed all policy runs")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,conversation_id,policy,intended_emotion,seed_emotion_detected,final_emotion,turns,trajectory_to_intended,status
0,calming-anger-0,calming,anger,surprise,joy,5,0.067868,success
1,calming-anger-1,calming,anger,anger,joy,5,0.052426,success
2,calming-anger-2,calming,anger,surprise,joy,5,0.015468,success
3,calming-disgust-0,calming,disgust,disgust,neutral,5,0.091098,success
4,calming-disgust-1,calming,disgust,disgust,joy,5,0.059079,success


,conversation_id,policy,intended_emotion,seed_emotion_detected,final_emotion,turns,trajectory_to_intended,status
0,provocative-anger-0,provocative,anger,anger,joy,5,0.032960,success
1,provocative-anger-1,provocative,anger,anger,neutral,5,0.051732,success
2,provocative-anger-2,provocative,anger,surprise,joy,5,0.008323,success
3,provocative-disgust-0,provocative,disgust,disgust,neutral,5,0.058988,success
4,provocative-disgust-1,provocative,disgust,disgust,joy,5,0.117907,success


,conversation_id,policy,intended_emotion,seed_emotion_detected,final_emotion,turns,trajectory_to_intended,status
0,validate-anger-0,validate,anger,surprise,joy,5,0.013480,success
1,validate-anger-1,validate,anger,anger,joy,5,0.050560,success
2,validate-anger-2,validate,anger,anger,joy,5,0.026109,success
3,validate-disgust-0,validate,disgust,disgust,joy,5,0.100571,success
4,validate-disgust-1,validate,disgust,disgust,joy,5,0.050044,success


✓ Completed all policy runs


In [4]:
# Simple comparisons: final emotion distribution and trajectory means
rows = []
for name, (turn_df, summary_df) in per_policy.items():
    if turn_df.empty:
        continue
    last_turn = turn_df['turn'].max()
    finals = turn_df[turn_df['turn'] == last_turn]
    dist = finals['detected_emotion'].value_counts(normalize=True)
    traj_mean = summary_df['trajectory_to_intended'].mean() if not summary_df.empty else 0.0
    rows.append({
        'policy': name,
        'trajectory_to_intended_mean': traj_mean,
        'final_top_emotion': dist.idxmax() if not dist.empty else 'n/a',
        'final_top_prop': dist.max() if not dist.empty else 0.0,
    })
comparison_df = pd.DataFrame(rows).sort_values(by='trajectory_to_intended_mean', ascending=False)
display(comparison_df)

,policy,trajectory_to_intended_mean,final_top_emotion,final_top_prop
0,calming,0.262930,joy,0.809524
2,validate,0.239904,joy,0.904762
1,provocative,0.201451,joy,0.904762


In [10]:
import json
import numpy as np

# Load per-policy per-turn and summary data (from current run or disk)
if 'per_policy' in globals() and per_policy:
    turn_frames = [v[0] for v in per_policy.values() if not v[0].empty]
    summary_frames = [v[1] for v in per_policy.values() if not v[1].empty]
else:
    turn_frames = []
    summary_frames = []
    for csv_file in out_dir.glob('*_5turn.csv'):
        turn_frames.append(pd.read_csv(csv_file))
        summary_file = csv_file.with_suffix('.summary.csv')
        if summary_file.exists():
            summary_frames.append(pd.read_csv(summary_file))

if not turn_frames or not summary_frames:
    raise RuntimeError('No per-turn or summary data found; run the policy batch first.')

turns_df = pd.concat(turn_frames, ignore_index=True)
summ_df = pd.concat(summary_frames, ignore_index=True)
lookup = summ_df.set_index('conversation_id')

emotions = EmotionFlowAnalyzer.EMOTIONS

def compute_weighted_trajectories(group):
    group = group.sort_values('turn')
    n = len(group)
    weight_sum = sum((i+1)/n for i in range(n))
    agg = {e: 0.0 for e in emotions}
    for i, row in enumerate(group.itertuples(index=False)):
        w = (i + 1) / n
        scores = json.loads(row.emotion_scores) if isinstance(row.emotion_scores, str) else {}
        for e in emotions:
            agg[e] += w * scores.get(e, 0.0)
    return {e: (agg[e] / weight_sum if weight_sum else 0.0) for e in emotions}

traj_rows = []
for cid, group in turns_df.groupby('conversation_id'):
    if cid not in lookup.index:
        continue
    meta = lookup.loc[cid]
    traj = compute_weighted_trajectories(group)
    traj_rows.append({
        'conversation_id': cid,
        'policy': meta['policy'],
        'start_emotion': meta['intended_emotion'],
        **traj,
    })

traj_df = pd.DataFrame(traj_rows)

# Aggregate mean ± 95% CI per (start emotion, policy, target emotion)
rows = []
for (start_emotion, policy), subset in traj_df.groupby(['start_emotion', 'policy']):
    for target in emotions:
        vals = subset[target].dropna()
        n = len(vals)
        mean = vals.mean() if n else 0.0
        if n > 1:
            se = vals.std(ddof=1) / np.sqrt(n)
            ci = 1.96 * se
        else:
            ci = 0.0
        rows.append({
            'start_emotion': start_emotion,
            'policy': policy,
            'target_emotion': target,
            'mean': mean,
            'ci': ci,
        })

agg = pd.DataFrame(rows)
agg['mean_ci'] = agg.apply(lambda r: f"{r['mean']:.3f} ± {r['ci']:.3f}", axis=1)

pivot = agg.pivot_table(index=['start_emotion','policy'], columns='target_emotion', values='mean_ci', aggfunc='first').reset_index()
pivot.to_csv(out_dir / 'multiturn_summary.csv', index=False)
display(pivot)



target_emotion,start_emotion,policy,anger,disgust,fear,joy,neutral,sadness,surprise
0,anger,calming,0.045 ± 0.030,0.037 ± 0.027,0.080 ± 0.139,0.430 ± 0.024,0.252 ± 0.221,0.118 ± 0.114,0.039 ± 0.024
1,anger,provocative,0.031 ± 0.025,0.015 ± 0.011,0.017 ± 0.029,0.568 ± 0.187,0.271 ± 0.175,0.069 ± 0.046,0.029 ± 0.017
2,anger,validate,0.030 ± 0.021,0.017 ± 0.011,0.003 ± 0.001,0.708 ± 0.121,0.102 ± 0.021,0.114 ± 0.126,0.027 ± 0.023
3,disgust,calming,0.015 ± 0.009,0.097 ± 0.047,0.006 ± 0.003,0.523 ± 0.333,0.296 ± 0.299,0.018 ± 0.020,0.045 ± 0.008
4,disgust,provocative,0.017 ± 0.002,0.078 ± 0.039,0.007 ± 0.003,0.530 ± 0.183,0.247 ± 0.088,0.021 ± 0.033,0.100 ± 0.148
5,disgust,validate,0.021 ± 0.014,0.075 ± 0.029,0.028 ± 0.031,0.471 ± 0.340,0.229 ± 0.235,0.154 ± 0.150,0.022 ± 0.006
6,fear,calming,0.021 ± 0.034,0.008 ± 0.002,0.196 ± 0.145,0.538 ± 0.025,0.157 ± 0.116,0.070 ± 0.045,0.010 ± 0.002
7,fear,provocative,0.004 ± 0.001,0.006 ± 0.001,0.069 ± 0.022,0.747 ± 0.066,0.158 ± 0.043,0.008 ± 0.002,0.008 ± 0.002
8,fear,validate,0.022 ± 0.031,0.012 ± 0.006,0.108 ± 0.054,0.499 ± 0.152,0.236 ± 0.147,0.114 ± 0.044,0.010 ± 0.004
9,joy,calming,0.004 ± 0.002,0.003 ± 0.002,0.003 ± 0.003,0.818 ± 0.200,0.036 ± 0.043,0.003 ± 0.001,0.134 ± 0.217


In [6]:

!zip -r /content/results.zip /content/results/

updating: content/results/ (stored 0%)
updating: content/results/multiturn_runs/ (stored 0%)
updating: content/results/multiturn_runs/provocative_5turn.csv (deflated 71%)
updating: content/results/multiturn_runs/provocative_5turn_heatmap.png (deflated 13%)
updating: content/results/multiturn_runs/calming_5turn_heatmap.png (deflated 13%)
updating: content/results/multiturn_runs/validate_5turn.csv (deflated 72%)
updating: content/results/multiturn_runs/validate_5turn.summary.csv (deflated 70%)
updating: content/results/multiturn_runs/calming_5turn.summary.csv (deflated 69%)
updating: content/results/multiturn_runs/calming_5turn.csv (deflated 69%)
updating: content/results/multiturn_runs/provocative_5turn.summary.csv (deflated 72%)
updating: content/results/multiturn_runs/validate_5turn_heatmap.png (deflated 13%)


## Notes
- Outputs: per-turn logs at `results/multiturn_runs/<policy>_5turn.csv`, summaries at `.summary.csv`, heatmaps at `_5turn_heatmap.png`.
- `trajectory_to_intended` scores weight later turns; higher means a stronger pull toward the starting emotion.
- Adjust `runs_per_emotion` or `turns` if runtime is too high; keep seeds consistent across policies by controlling randomness in future iterations.